# Deployment
## Setup

In [2]:
from dotenv import load_dotenv
from pathlib import Path
import os

In [3]:
load_dotenv()  # Load environment variables from .env file
PARENT = Path(os.getcwd()).parent

DATA_DIR_PROCESSED = Path(os.getenv("DATA_DIR_PROCESSED"))
DATA_DIR_PROCESSED = PARENT / DATA_DIR_PROCESSED

MODEL_DIR = Path(os.getenv("MODEL_DIR"))
MODEL_DIR = PARENT / MODEL_DIR

## API Testing
This section tests all endpoints in the Flask API for the Loan Default Prediction model.

In [ ]:
import json
import sys
from pathlib import Path

# Add src directory to path for imports
sys.path.insert(0, str(PARENT / 'src'))

print("Testing Loan Default Prediction API using Flask Test Client")

Testing Loan Default Prediction API


### Step 1: Initialize the Test Client
Set up the Flask test client for API testing. This approach tests the endpoints without requiring an external server.

In [27]:
import os
from src.app import app

# Create a Flask test client
client = app.test_client()

print("✓ Flask test client initialized successfully!")
print("Ready to test API endpoints...")

✓ Flask test client initialized successfully!
Ready to test API endpoints...


### Step 2: Test Home Endpoint (GET /)
Retrieve API documentation and available endpoints

In [19]:
response = client.get("/")
print(f"Status Code: {response.status_code}")
print(f"Response:\n{json.dumps(response.get_json(), indent=2)}")

Status Code: 200
Response:
{
  "endpoints": {
    "GET /health": "Check API health status",
    "POST /predict": "Make a prediction on a single loan application",
    "POST /predict-batch": "Make predictions on multiple loan applications"
  },
  "message": "Loan Default Prediction API",
  "version": "1.0.0"
}


### Step 3: Test Health Check Endpoint (GET /health)
Check if the API and models are ready for predictions

In [20]:
response = client.get("/health")
print(f"Status Code: {response.status_code}")
print(f"Response:\n{json.dumps(response.get_json(), indent=2)}")

Status Code: 200
Response:
{
  "message": "API is running and models are loaded",
  "status": "healthy"
}


### Step 4: Test Single Prediction Endpoint (POST /predict)
Make a prediction on a single loan application

In [21]:
# Sample loan application data
sample_loan = {
    "credit_policy": 1,
    "purpose": "debt_consolidation",
    "interest_rate": 0.11,
    "installment": 829.67,
    "log_annual_income": 10.5,
    "debt_income_ratio": 0.22,
    "fico": 700,
    "days_with_credit_line": 4980,
    "revolve_balance": 5598,
    "revolve_utilized": 0.45,
    "inquiries_last_6_mon": 1,
    "delinquent_2_yrs": 0,
    "public_recs": 0
}

print("Testing single prediction with sample data:")
print(json.dumps(sample_loan, indent=2))
print("\n" + "="*50 + "\n")

response = client.post("/predict", json=sample_loan)
print(f"Status Code: {response.status_code}")
print(f"Response:\n{json.dumps(response.get_json(), indent=2)}")

Testing single prediction with sample data:
{
  "credit_policy": 1,
  "purpose": "debt_consolidation",
  "interest_rate": 0.11,
  "installment": 829.67,
  "log_annual_income": 10.5,
  "debt_income_ratio": 0.22,
  "fico": 700,
  "days_with_credit_line": 4980,
  "revolve_balance": 5598,
  "revolve_utilized": 0.45,
  "inquiries_last_6_mon": 1,
  "delinquent_2_yrs": 0,
  "public_recs": 0
}


Status Code: 200
Response:
{
  "input": {
    "credit_policy": 1,
    "days_with_credit_line": 4980,
    "debt_income_ratio": 0.22,
    "delinquent_2_yrs": 0,
    "fico": 700,
    "inquiries_last_6_mon": 1,
    "installment": 829.67,
    "interest_rate": 0.11,
    "log_annual_income": 10.5,
    "public_recs": 0,
    "purpose": "debt_consolidation",
    "revolve_balance": 5598,
    "revolve_utilized": 0.45
  },
  "prediction": {
    "default_risk": "High",
    "prediction": 1,
    "probability_default": 1.0,
    "probability_no_default": 0.0
  },
  "success": true
}


### Step 5: Test Batch Prediction Endpoint (POST /predict-batch)
Make predictions on multiple loan applications at once

In [22]:
# Create multiple sample loans with different risk profiles
sample_batch = {
    "records": [
        {
            "credit_policy": 1,
            "purpose": "debt_consolidation",
            "interest_rate": 0.11,
            "installment": 829.67,
            "log_annual_income": 10.5,
            "debt_income_ratio": 0.22,
            "fico": 700,
            "days_with_credit_line": 4980,
            "revolve_balance": 5598,
            "revolve_utilized": 0.45,
            "inquiries_last_6_mon": 1,
            "delinquent_2_yrs": 0,
            "public_recs": 0
        },
        {
            "credit_policy": 1,
            "purpose": "credit_card",
            "interest_rate": 0.15,
            "installment": 1200.00,
            "log_annual_income": 10.2,
            "debt_income_ratio": 0.35,
            "fico": 650,
            "days_with_credit_line": 3000,
            "revolve_balance": 8000,
            "revolve_utilized": 0.85,
            "inquiries_last_6_mon": 3,
            "delinquent_2_yrs": 1,
            "public_recs": 0
        },
        {
            "credit_policy": 1,
            "purpose": "home_improvement",
            "interest_rate": 0.08,
            "installment": 500.00,
            "log_annual_income": 11.0,
            "debt_income_ratio": 0.15,
            "fico": 750,
            "days_with_credit_line": 8000,
            "revolve_balance": 2000,
            "revolve_utilized": 0.20,
            "inquiries_last_6_mon": 0,
            "delinquent_2_yrs": 0,
            "public_recs": 0
        }
    ]
}

print("Testing batch predictions with 3 sample loan applications:")
print(f"Total records: {len(sample_batch['records'])}")
print("\n" + "="*50 + "\n")

response = client.post("/predict-batch", json=sample_batch)
print(f"Status Code: {response.status_code}")
result = response.get_json()
print(f"Response:\n{json.dumps(result, indent=2)}")

Testing batch predictions with 3 sample loan applications:
Total records: 3


Status Code: 200
Response:
{
  "errors": null,
  "failed_predictions": 0,
  "predictions": [
    {
      "prediction": {
        "default_risk": "High",
        "prediction": 1,
        "probability_default": 1.0,
        "probability_no_default": 0.0
      },
      "record_index": 0
    },
    {
      "prediction": {
        "default_risk": "High",
        "prediction": 1,
        "probability_default": 1.0,
        "probability_no_default": 0.0
      },
      "record_index": 1
    },
    {
      "prediction": {
        "default_risk": "Low",
        "prediction": 0,
        "probability_default": 0.0,
        "probability_no_default": 1.0
      },
      "record_index": 2
    }
  ],
  "success": true,
  "successful_predictions": 3,
  "total_records": 3
}


### Step 6: Test Error Handling
Test various error scenarios and edge cases

In [23]:
print("Test 1: Missing required features")
print("-" * 50)
incomplete_data = {
    "credit_policy": 1,
    "purpose": "debt_consolidation",
    "interest_rate": 0.11
    # Missing other required features
}
response = client.post("/predict", json=incomplete_data)
print(f"Status Code: {response.status_code}")
print(f"Response:\n{json.dumps(response.get_json(), indent=2)}\n")

Test 1: Missing required features
--------------------------------------------------
Status Code: 400
Response:
{
  "error": "Missing required features: installment, log_annual_income, debt_income_ratio, fico, days_with_credit_line, revolve_balance, revolve_utilized, inquiries_last_6_mon, delinquent_2_yrs, public_recs"
}



In [24]:
print("\nTest 2: Invalid endpoint (404 error)")
print("-" * 50)
response = client.get("/invalid-endpoint")
print(f"Status Code: {response.status_code}")
print(f"Response:\n{json.dumps(response.get_json(), indent=2)}\n")


Test 2: Invalid endpoint (404 error)
--------------------------------------------------
Status Code: 404
Response:
{
  "error": "Endpoint not found",
  "message": "Use GET / to see available endpoints"
}



In [25]:
print("\nTest 3: Empty JSON data")
print("-" * 50)
response = client.post("/predict", json={})
print(f"Status Code: {response.status_code}")
print(f"Response:\n{json.dumps(response.get_json(), indent=2)}\n")


Test 3: Empty JSON data
--------------------------------------------------
Status Code: 400
Response:
{
  "error": "No JSON data provided"
}



In [26]:
print("\nTest 4: Invalid batch request (missing 'records' key)")
print("-" * 50)
response = client.post("/predict-batch", json={"data": []})
print(f"Status Code: {response.status_code}")
print(f"Response:\n{json.dumps(response.get_json(), indent=2)}")


Test 4: Invalid batch request (missing 'records' key)
--------------------------------------------------
Status Code: 400
Response:
{
  "error": "Expected JSON with \"records\" key containing list of loan applications"
}


### Step 7: Summary
API testing completed. The following endpoints have been tested:

1. **GET /** - Home endpoint with API documentation ✓
2. **GET /health** - Health check to verify API and models are ready ✓
3. **POST /predict** - Single loan prediction with all required features ✓
4. **POST /predict-batch** - Batch predictions for multiple loans ✓
5. **Error Handling** - Missing features, invalid endpoints, and malformed requests ✓

All endpoints are working as expected!